In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
import numpy as np
import tensorflow as tf

from tensorflow.keras.models import load_model, Model
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam

In [41]:
TRAIN_DIR = "/content/drive/MyDrive/disaster_final/train"
VAL_DIR = "/content/drive/MyDrive/disaster_final/validation"
TEST_DIR = "/content/drive/MyDrive/disaster_final/test"

IMG_SIZE = 224
BATCH_SIZE = 16
EPOCHS = 35

In [42]:
DISASTER_TYPES = ["earthquake","flood","wildfire"]
SEVERITY_LEVELS = ["high","low","medium"]

SEVERITY_LABEL = {
    "high":0,
    "medium":1,
    "low":2
}

In [43]:
def load_severity_paths(dataset_dir):

    paths=[]
    labels=[]

    for disaster in DISASTER_TYPES:
        for severity in SEVERITY_LEVELS:

            folder=os.path.join(dataset_dir,disaster,severity)

            if not os.path.exists(folder):
                continue

            for file in os.listdir(folder):

                if file.lower().endswith((".jpg",".jpeg",".png")):

                    paths.append(os.path.join(folder,file))
                    labels.append(SEVERITY_LABEL[severity])

    return np.array(paths),np.array(labels)

In [53]:
train_paths,train_labels = load_severity_paths(TRAIN_DIR)
val_paths,val_labels = load_severity_paths(VAL_DIR)
test_paths,test_labels = load_severity_paths(TEST_DIR)

print("Train:",len(train_paths))
print("Validation:",len(val_paths))
print("Test:",len(test_paths))

Train: 2244
Validation: 892
Test: 462


In [54]:
def load_image(path,label):

    img=tf.io.read_file(path)
    img=tf.io.decode_image(img,channels=3)

    img.set_shape([None,None,3])

    img=tf.image.resize(img,[IMG_SIZE,IMG_SIZE])

    # augmentation
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_brightness(img, 0.1)
    img = tf.image.random_contrast(img, 0.9, 1.1)

    img=tf.cast(img,tf.float32)/255.0

    return img,label

In [55]:
train_ds=tf.data.Dataset.from_tensor_slices((train_paths,train_labels))

train_ds=train_ds.map(load_image)\
                 .shuffle(2000)\
                 .batch(BATCH_SIZE)\
                 .prefetch(tf.data.AUTOTUNE)

val_ds=tf.data.Dataset.from_tensor_slices((val_paths,val_labels))

val_ds=val_ds.map(load_image)\
             .batch(BATCH_SIZE)

In [56]:
hera_model = load_model(
"/content/drive/MyDrive/classifier_model_V_Hera_fixed2.0.keras"
)

print("Updated Hera loaded")

Updated Hera loaded


In [57]:
# Freeze Zeus backbone
hera_model.layers[0].trainable = False

# Freeze early Hera layers
for layer in hera_model.layers[1:5]:
    layer.trainable = False

# Train only last Hera layers
for layer in hera_model.layers[5:]:
    layer.trainable = True

In [61]:
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.models import Model

# Hera feature layer (before classifier)
features = hera_model.layers[-2].output   # dense_12

x = Dense(256, activation="relu", name="ares_dense1")(features)
x = BatchNormalization(name="ares_bn1")(x)
x = Dropout(0.4, name="ares_drop1")(x)

x = Dense(128, activation="relu", name="ares_dense2")(x)

severity_output = Dense(3, activation="softmax", name="ares_output")(x)

ares_model = Model(
    inputs=hera_model.input,
    outputs=severity_output,
    name="severity_model_V_Ares"
)

ares_model.summary()

Model: "severity_model_V_Ares"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 512)            │       655,872 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ ares_dense1 (Dense)             │ (None, 256)            │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ ares_bn1 (BatchNormalization)   │ (None, 256)            │         1,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ ares_drop1 (Dropout)            │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ ares_dense2 (Dense)             │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ ares_output (Dense)             │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,940,070 (18.84 MB)

 Trainable params: 231,555 (904.51 KB)

 Non-trainable params: 4,708,515 (17.96 MB)

In [62]:
for layer in hera_model.layers[-20:]:
    layer.trainable = True

In [64]:
from tensorflow.keras.optimizers import Adam

ares_model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [65]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_accuracy",
    patience=5,
    restore_best_weights=True
)

ares_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=[early_stop]
)

Epoch 1/35
141/141 ━━━━━━━━━━━━━━━━━━━━ 199s 608ms/step - accuracy: 0.3445 - loss: 1.5474 - val_accuracy: 0.5448 - val_loss: 0.9456
Epoch 2/35
141/141 ━━━━━━━━━━━━━━━━━━━━ 58s 201ms/step - accuracy: 0.4689 - loss: 1.2048 - val_accuracy: 0.7567 - val_loss: 0.6785
Epoch 3/35
141/141 ━━━━━━━━━━━━━━━━━━━━ 57s 199ms/step - accuracy: 0.5716 - loss: 0.9745 - val_accuracy: 0.8217 - val_loss: 0.5087
Epoch 4/35
141/141 ━━━━━━━━━━━━━━━━━━━━ 56s 198ms/step - accuracy: 0.6417 - loss: 0.8183 - val_accuracy: 0.8677 - val_loss: 0.4120
Epoch 5/35
141/141 ━━━━━━━━━━━━━━━━━━━━ 58s 202ms/step - accuracy: 0.6869 - loss: 0.6993 - val_accuracy: 0.8834 - val_loss: 0.3624
Epoch 6/35
141/141 ━━━━━━━━━━━━━━━━━━━━ 56s 199ms/step - accuracy: 0.7665 - loss: 0.5966 - val_accuracy: 0.9081 - val_loss: 0.3105
Epoch 7/35
141/141 ━━━━━━━━━━━━━━━━━━━━ 56s 196ms/step - accuracy: 0.7729 - loss: 0.5502 - val_accuracy: 0.9114 - val_loss: 0.2933
Epoch 8/35
141/141 ━━━━━━━━━━━━━━━━━━━━ 56s 197ms/step - accuracy: 0.7814 - loss: 

In [66]:
test_ds=tf.data.Dataset.from_tensor_slices((test_paths,test_labels))

test_ds=test_ds.map(load_image)\
               .batch(BATCH_SIZE)

ares_model.evaluate(test_ds)

29/29 ━━━━━━━━━━━━━━━━━━━━ 16s 559ms/step - accuracy: 0.7511 - loss: 0.9744


[1.821373701095581, 0.5432900190353394]

In [67]:
import numpy as np
print(np.bincount(train_labels))

[749 749 746]


In [68]:
ares_model.save(
"/content/drive/MyDrive/Severity_classifier_model_V_Ares2.0.keras"
)

print("Ares model saved")

Ares model saved
